In [1]:
import polars as pl
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, LinearColorMapper, ColorBar, TapTool, OpenURL
from bokeh.io import output_notebook
from bokeh.palettes import Plasma256, Viridis256, Turbo10
import numpy as np

In [8]:
# title_episode = pl.read_csv(
#         raw_data_path,
#         has_header=True,
#         separator="\t",
#         truncate_ragged_lines=True,
#         null_values="\\N",
#         quote_char=None,
#         schema={
#             "tconst": pl.Utf8,
#             "parentTconst": pl.Utf8,
#             "seasonNumber": pl.UInt16,
#             "episodeNumber": pl.UInt32,
#         },
#     )

In [6]:
movies = pl.read_csv("data-1778850350675.csv", null_values="NULL")
movies = movies.filter(pl.col("start_year").is_not_null(), pl.col("genre").is_not_null())
movies

genre,start_year,average_rating,num_votes,primary_title,tconst,priority,title_type,genres,directors
str,i64,f64,i64,str,str,bool,str,str,str
"""Comedy""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Action""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Romance""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Action""",1926,8.1,106053,"""The General""","""tt0017925""",false,"""movie""","""Action - Adventure - Comedy""","""Buster Keaton - Clyde Bruckman"""
"""Adventure""",1926,8.1,106053,"""The General""","""tt0017925""",false,"""movie""","""Action - Adventure - Comedy""","""Buster Keaton - Clyde Bruckman"""
…,…,…,…,…,…,…,…,…,…
"""Sci-Fi""",2026,6.3,85248,"""War Machine""","""tt15940132""",false,"""movie""","""Action - Sci-Fi - Thriller""","""Patrick Hughes"""
"""Thriller""",2026,6.3,85248,"""War Machine""","""tt15940132""",false,"""movie""","""Action - Sci-Fi - Thriller""","""Patrick Hughes"""
"""Adventure""",2026,8.4,195948,"""Project Hail Mary""","""tt12042730""",false,"""movie""","""Adventure - Comedy - Sci-Fi""","""Phil Lord - Christopher Miller"""


In [16]:
movies["priority"].value_counts().filter(pl.col("priority") == "true")["count"].item()
# .value_counts()
    #     .filter(pl.col("priority") == "true")["count"]
    #     .item()

254

In [17]:
movies

genre,start_year,average_rating,num_votes,primary_title,tconst,priority,title_type,genres,directors
str,i64,f64,i64,str,str,bool,str,str,str
"""Comedy""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Action""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Romance""",1924,8.1,65521,"""Sherlock Jr.""","""tt0015324""",false,"""movie""","""Action - Comedy - Romance""","""Buster Keaton"""
"""Action""",1926,8.1,106053,"""The General""","""tt0017925""",false,"""movie""","""Action - Adventure - Comedy""","""Buster Keaton - Clyde Bruckman"""
"""Adventure""",1926,8.1,106053,"""The General""","""tt0017925""",false,"""movie""","""Action - Adventure - Comedy""","""Buster Keaton - Clyde Bruckman"""
…,…,…,…,…,…,…,…,…,…
"""Sci-Fi""",2026,6.3,85248,"""War Machine""","""tt15940132""",false,"""movie""","""Action - Sci-Fi - Thriller""","""Patrick Hughes"""
"""Thriller""",2026,6.3,85248,"""War Machine""","""tt15940132""",false,"""movie""","""Action - Sci-Fi - Thriller""","""Patrick Hughes"""
"""Adventure""",2026,8.4,195948,"""Project Hail Mary""","""tt12042730""",false,"""movie""","""Adventure - Comedy - Sci-Fi""","""Phil Lord - Christopher Miller"""


In [4]:
movies = movies.with_columns(
    ("https://www.imdb.com/title/" + pl.col("tconst") + "/").alias("url")
)

In [5]:
output_notebook()

Loading BokehJS ...

In [9]:
# Scale num_votes to 0–10
movies = movies.with_columns(
    (
        (pl.col("num_votes") - pl.col("num_votes").min()) /
        (pl.col("num_votes").max() - pl.col("num_votes").min())
        * 1.5 + 1.2
    ).alias("votes_scaled"),
    (np.log(pl.col("start_year")) * 3000).alias("year_log")
)

source = ColumnDataSource(movies)

color_mapper = LinearColorMapper(
    palette=Turbo10,
    low=movies["average_rating"].min(),
    high=movies["average_rating"].max(),
)

genres = movies["genre"].unique().sort(descending=True).to_list()
unique_years = sorted(movies["start_year"].unique())
decade_years = [y for y in unique_years if y % 10 == 0]
tooltips = [("Title", "@primary_title"), ("Year", "@start_year"), ("Selected genre", "@genre"), ("Directors", "@directors"),("Genres", "@genres"), ("Average Rating", "@average_rating"), ("# votes", "@num_votes")]

p = figure(
    height=400,
    width=1000,
    y_range=genres,
    tooltips=tooltips,
    tools="tap, box_zoom, wheel_zoom, reset, pan, hover, save",
    # x_axis_type="log"
    )
p.circle(
    x="year_log",
    y="genre",
    radius="votes_scaled",
    alpha=0.5,
    color={"field": "average_rating", "transform": color_mapper},
    source=source,
)
p.select_one(TapTool).callback = OpenURL(url="@url")
color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12)
# tick_positions = { 
#     np.log(y) * 3000: str(y) 
#     for y in decade_years
# }

taptool = p.select_one(TapTool)
taptool.callback = OpenURL(url="@url")


# p.xaxis.ticker = list(tick_positions.keys())
# p.xaxis.major_label_overrides = tick_positions

p.add_layout(color_bar, "right")




show(p)

In [13]:
# Scale num_votes to 0–10
movies = movies.with_columns(
    (
        (pl.col("num_votes") - pl.col("num_votes").min()) /
        (pl.col("num_votes").max() - pl.col("num_votes").min())
        * 1.5 + 1.2
    ).alias("votes_scaled"),
)

source = ColumnDataSource(movies)

color_mapper = LinearColorMapper(
    palette=Turbo10,
    low=movies["average_rating"].min(),
    high=movies["average_rating"].max(),
)

genres = movies["genre"].unique().sort(descending=True).to_list()
tooltips = [("Title", "@primary_title"), ("Year", "@start_year"), ("Selected genre", "@genre"), ("Directors", "@directors"),("Genres", "@genres"), ("Average Rating", "@average_rating"), ("# votes", "@num_votes")]

p = figure(
    height=400,
    width=1000,
    y_range=genres,
    tooltips=tooltips,
    tools="tap, box_zoom, wheel_zoom, reset, pan, hover, save",
    x_axis_type="log"
    )
p.circle(
    x="start_year",
    y="genre",
    radius="votes_scaled",
    alpha=0.5,
    color={"field": "average_rating", "transform": color_mapper},
    source=source,
)
p.select_one(TapTool).callback = OpenURL(url="@url")
# p.xaxis.axis_label = "release year"
# p.yaxis.axis_label = "genres"
color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12)

taptool = p.select_one(TapTool)
taptool.callback = OpenURL(url="@url")

p.add_layout(color_bar, "right")


show(p)